# 10 - Resumen Ejecutivo
## Dashboard del Pipeline de Bioactividad - PubChem

---

### Objetivo de este notebook

Este notebook presenta un **resumen ejecutivo** del pipeline ELT completo,
consolidando todos los indicadores clave de rendimiento (KPIs), el estado
de cada capa y los hallazgos principales.

### Contenido del resumen

1. **KPIs del Pipeline** - Metricas clave de volumetria y rendimiento
2. **Estado de las Capas** - Disponibilidad y conteo por capa
3. **Top Compuestos** - Medicamentos con mayor actividad biologica
4. **Analisis Lipinski** - Cumplimiento de la regla de biodisponibilidad
5. **Distribucion de Resultados** - Proporcion activos vs inactivos
6. **Resumen para Presentacion** - Ficha tecnica del proyecto

### Publico objetivo

Este notebook esta disenado para ser presentado a:
- Directores de proyecto y tomadores de decisiones
- Equipos de ingenieria de datos para revision tecnica
- Stakeholders de investigacion farmacologica

---
## 1. Configuracion Inicial

In [ ]:
from pyspark.sql import functions as F  # Funciones de transformacion de Spark SQL
from datetime import datetime            # Para generar fecha del reporte

In [ ]:
# Configuracion del catalogo activo
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]

# Encabezado del reporte
print(f"Pipeline de Bioactividad PubChem - Resumen Ejecutivo")
print(f"Catalogo: {CATALOGO}")
print(f"Fecha del reporte: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. Indicadores Clave de Rendimiento (KPIs)

Se recopilan las metricas mas importantes de cada capa del pipeline.
Estos KPIs permiten evaluar rapidamente si el pipeline funciono
correctamente y cuantos datos se procesaron.

La **tasa de retencion** (Bronce -> Plata) indica que porcentaje de datos
crudos superaron las validaciones y transformaciones de calidad.

In [ ]:
# Diccionario para almacenar metricas de cada capa
metricas = {}

# --- Recopilar metricas de la capa Bronce ---
try:
    metricas["bronce_compuestos"] = spark.table("pubchem_bronce.propiedades_compuestos").count()
    metricas["bronce_bioactividad"] = spark.table("pubchem_bronce.bioactividad").count()
except:
    metricas["bronce_compuestos"] = 0
    metricas["bronce_bioactividad"] = 0

# --- Recopilar metricas de la capa Plata ---
try:
    metricas["plata_actividades"] = spark.table("pubchem_plata.actividades_biologicas").count()
except:
    metricas["plata_actividades"] = 0

# --- Recopilar metricas de la capa Oro ---
try:
    metricas["oro_compuestos"] = spark.table("pubchem_oro.dim_compuestos").count()
    metricas["oro_ensayos"] = spark.table("pubchem_oro.dim_ensayos").count()
    metricas["oro_resultados"] = spark.table("pubchem_oro.dim_resultados").count()
    metricas["oro_hechos"] = spark.table("pubchem_oro.fact_bioactividades").count()
except:
    metricas["oro_compuestos"] = 0
    metricas["oro_ensayos"] = 0
    metricas["oro_resultados"] = 0
    metricas["oro_hechos"] = 0

# --- Mostrar KPIs ---
print("=" * 70)
print("              INDICADORES CLAVE DE RENDIMIENTO (KPIs)")
print("=" * 70)
print()
print(f"  Medicamentos analizados:           {metricas['bronce_compuestos']:>10,}")
print(f"  Ensayos biologicos unicos:         {metricas['oro_ensayos']:>10,}")
print(f"  Registros de bioactividad (Bronce): {metricas['bronce_bioactividad']:>10,}")
print(f"  Registros procesados (Plata):      {metricas['plata_actividades']:>10,}")
print(f"  Hechos en modelo estrella (Oro):   {metricas['oro_hechos']:>10,}")
print(f"  Tipos de resultado:                {metricas['oro_resultados']:>10,}")

# Calcular y mostrar tasa de retencion si hay datos
if metricas["bronce_bioactividad"] > 0:
    tasa_retencion = round(
        metricas["plata_actividades"] * 100 / metricas["bronce_bioactividad"], 1
    )
    print(f"  Tasa de retencion (Bronce->Plata): {tasa_retencion:>9.1f}%")

print()
print("=" * 70)

---
## 3. Estado de las Capas del Pipeline

Verificacion detallada del estado de cada capa de la arquitectura medallon:
- **Raw**: Archivos JSON crudos en el volumen de Unity Catalog
- **Bronce**: Delta Tables con datos sin transformar
- **Plata**: Datos limpios, estandarizados y enriquecidos
- **Oro**: Modelo estrella optimizado para consultas analiticas

In [ ]:
# Encabezado del estado de capas
print("=" * 70)
print("              ESTADO DE CAPAS DEL PIPELINE")
print("=" * 70)
print()

# Definicion de las capas con su descripcion
capas = [
    ("RAW",    "Archivos JSON crudos del API de PubChem"),
    ("BRONCE", "Delta Tables sin transformaciones de negocio"),
    ("PLATA",  "Datos limpios, estandarizados y enriquecidos"),
    ("ORO",    "Modelo estrella optimizado para analisis"),
]

# Tablas asociadas a cada capa para verificacion
tablas_por_capa = {
    "RAW": [],  # La capa Raw se verifica por archivos, no por tablas
    "BRONCE": [
        "pubchem_bronce.propiedades_compuestos",
        "pubchem_bronce.bioactividad"
    ],
    "PLATA": [
        "pubchem_plata.actividades_biologicas"
    ],
    "ORO": [
        "pubchem_oro.dim_compuestos",
        "pubchem_oro.dim_ensayos",
        "pubchem_oro.dim_resultados",
        "pubchem_oro.fact_bioactividades"
    ]
}

# Verificar cada capa
for capa, descripcion in capas:
    print(f"  [{capa}] {descripcion}")

    if capa == "RAW":
        # Verificar capa Raw contando archivos en el volumen
        RUTA_RAW = f"/Volumes/{CATALOGO}/pubchem_raw/bioactividad"
        try:
            archivos = dbutils.fs.ls(RUTA_RAW)
            print(f"    Estado: DISPONIBLE ({len(archivos)} archivos)")
        except:
            print(f"    Estado: NO DISPONIBLE")
    else:
        # Verificar capas tabulares contando registros de cada tabla
        for tabla in tablas_por_capa[capa]:
            try:
                conteo = spark.table(tabla).count()
                print(f"    {tabla}: {conteo:,} registros")
            except:
                print(f"    {tabla}: NO DISPONIBLE")

    print()  # Linea en blanco entre capas

---
## 4. Top Compuestos por Actividad Biologica

Ranking de los 15 medicamentos por cantidad de ensayos biologicos
y tasa de actividad. Incluye la clasificacion Lipinski para
evaluar la relacion entre biodisponibilidad y actividad.

In [ ]:
# Consulta de ranking de compuestos por actividad biologica
print("--- Top compuestos por cantidad de ensayos y tasa de actividad ---")
print()

spark.sql("""
    SELECT
        dc.nombre_comun AS medicamento,               -- Nombre del medicamento
        dc.clasificacion_lipinski AS lipinski,         -- Clasificacion Lipinski
        COUNT(*) AS total_ensayos,                    -- Total de ensayos
        SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) AS activos,
        SUM(CASE WHEN dr.resultado_actividad = 'Inactive' THEN 1 ELSE 0 END) AS inactivos,
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            1
        ) AS tasa_activos_pct                         -- Tasa de actividad
    FROM pubchem_oro.fact_bioactividades f
    JOIN pubchem_oro.dim_compuestos dc ON f.llave_compuesto = dc.llave_compuesto
    JOIN pubchem_oro.dim_resultados dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dc.nombre_comun, dc.clasificacion_lipinski
    ORDER BY total_ensayos DESC
""").show(truncate=False)

---
## 5. Analisis de Cumplimiento de la Regla de Lipinski

La Regla de los 5 de Lipinski predice la biodisponibilidad oral de un compuesto.
Este analisis muestra:
1. Cuantos de los 15 medicamentos cumplen los criterios
2. El detalle de propiedades de cada compuesto
3. Comparacion de tasa de actividad entre los que cumplen y no cumplen

In [ ]:
# Encabezado del analisis de Lipinski
print("=" * 70)
print("         CUMPLIMIENTO DE LA REGLA DE LOS 5 DE LIPINSKI")
print("=" * 70)
print()

# Cargar dimension de compuestos para el analisis
df_lipinski = spark.table("pubchem_oro.dim_compuestos")

# Calcular conteos por clasificacion
total_comp = df_lipinski.count()
cumplen = df_lipinski.filter(F.col("clasificacion_lipinski") == "Cumple Lipinski").count()
no_cumplen = total_comp - cumplen

# Mostrar resumen de cumplimiento
print(f"  Total de compuestos analizados:   {total_comp}")
print(f"  Cumplen Regla de Lipinski:        {cumplen} ({round(cumplen * 100 / total_comp, 1)}%)")
print(f"  No cumplen Regla de Lipinski:     {no_cumplen} ({round(no_cumplen * 100 / total_comp, 1)}%)")
print()

# Detalle por compuesto con propiedades relevantes
print("--- Detalle de compuestos por clasificacion ---")
df_lipinski.select(
    "nombre_comun",            # Nombre del medicamento
    "peso_molecular",          # Peso molecular (criterio Lipinski: <= 500)
    "xlogp",                   # Lipofilicidad (criterio Lipinski: <= 5)
    "clasificacion_lipinski"   # Resultado de la clasificacion
).orderBy("clasificacion_lipinski", "nombre_comun").show(truncate=False)

In [ ]:
# Comparar tasa de actividad entre compuestos que cumplen vs no cumplen Lipinski
# Esta comparacion permite evaluar si la biodisponibilidad oral
# se correlaciona con la actividad biologica medida en ensayos
print("--- Tasa de actividad: Cumple Lipinski vs No cumple ---")
print()

spark.sql("""
    SELECT
        dc.clasificacion_lipinski,                    -- Grupo de comparacion
        COUNT(*) AS total_ensayos,                    -- Total de ensayos por grupo
        SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) AS activos,
        ROUND(
            SUM(CASE WHEN dr.resultado_actividad = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS tasa_actividad_pct,                      -- Tasa de actividad del grupo
        ROUND(AVG(f.valor_actividad_um), 4) AS promedio_actividad_um  -- Potencia promedio
    FROM pubchem_oro.fact_bioactividades f
    JOIN pubchem_oro.dim_compuestos dc ON f.llave_compuesto = dc.llave_compuesto
    JOIN pubchem_oro.dim_resultados dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dc.clasificacion_lipinski
    ORDER BY clasificacion_lipinski
""").show(truncate=False)

---
## 6. Distribucion Global de Resultados

Visualizacion de la proporcion de cada tipo de resultado de bioactividad
en todo el dataset, con una barra de texto proporcional para
facilitar la comparacion visual.

In [ ]:
# Distribucion de resultados con barra visual de texto
# La funcion REPEAT genera una barra de asteriscos proporcional al porcentaje
print("--- Distribucion global de resultados de bioactividad ---")
print()

spark.sql("""
    SELECT
        dr.resultado_actividad,                       -- Tipo de resultado
        COUNT(*) AS cantidad,                         -- Cantidad de registros
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),  -- Porcentaje del total
            2
        ) AS porcentaje,
        REPEAT('*', CAST(
            COUNT(*) * 50 / SUM(COUNT(*)) OVER()      -- Barra visual proporcional
        AS INT)) AS barra
    FROM pubchem_oro.fact_bioactividades f
    JOIN pubchem_oro.dim_resultados dr ON f.llave_resultado = dr.llave_resultado
    GROUP BY dr.resultado_actividad
    ORDER BY cantidad DESC
""").show(truncate=False)

---
## 7. Ficha Tecnica del Proyecto

Resumen consolidado con toda la informacion del proyecto
para presentacion ejecutiva.

In [ ]:
# Resumen ejecutivo consolidado del pipeline
print("=" * 70)
print("         RESUMEN EJECUTIVO - PIPELINE PUBCHEM")
print("=" * 70)
print()
print("  PROYECTO: Pipeline ELT de Bioactividad de Medicamentos")
print("  FUENTE:   PubChem BioAssay (NIH - National Library of Medicine)")
print(f"  FECHA:    {datetime.now().strftime('%Y-%m-%d')}")
print()
print("  ARQUITECTURA:")
print("    - Plataforma:  Databricks con Unity Catalog")
print("    - Computo:     Serverless")
print("    - Formato:     Delta Lake")
print("    - Modelo:      Arquitectura Medallon (Raw -> Bronce -> Plata -> Oro)")
print()
print("  DATOS PROCESADOS:")
print(f"    - Medicamentos:              {metricas['bronce_compuestos']}")
print(f"    - Registros de bioactividad: {metricas['bronce_bioactividad']:,}")
print(f"    - Ensayos biologicos unicos: {metricas['oro_ensayos']:,}")
print(f"    - Registros en modelo final: {metricas['oro_hechos']:,}")
print()
print("  MODELO DIMENSIONAL:")
print(f"    - Dimensiones: 3 (compuestos, ensayos, resultados)")
print(f"    - Tabla de hechos: 1 (fact_bioactividades)")
print(f"    - Total tablas Oro: 4")
print()
print("  CALIDAD:")
print(f"    - Compuestos Lipinski: {cumplen}/{total_comp} cumplen la Regla de los 5")
if metricas["bronce_bioactividad"] > 0:
    print(f"    - Retencion de datos:  {tasa_retencion}% (Bronce -> Plata)")
print()
print("  NOTEBOOKS DEL PIPELINE:")
print("    00 - Configuracion inicial")
print("    01 - Ingesta desde PubChem API (Raw)")
print("    02 - Carga Raw a Bronce (Delta Tables)")
print("    03 - Exploracion y perfilamiento")
print("    04 - Validacion de calidad (Quality Gate)")
print("    05 - Transformacion Bronce a Plata")
print("    06 - Modelo estrella Plata a Oro")
print("    07 - Orquestacion del pipeline")
print("    08 - Logs de auditoria")
print("    09 - Consultas analiticas")
print("    10 - Resumen ejecutivo (este notebook)")
print()
print("=" * 70)